# Notebook 04: Baseline Analysis and Ablations

Three analyses to strengthen the baseline before fine-tuning:

1. **Additional embedding models** (text-embedding-3-large)
2. **API format ablation** (which parts of the API string matter?)
3. **Error analysis** (where does the baseline fail?)

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))

from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_random_negatives, build_category_sibling_negatives, build_dfsdt_negatives
from models.embeddings import format_api_string, get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import recall_at_k, mean_reciprocal_rank, evaluate_batch

corpus = load_api_corpus(TOOLBENCH_DIR / 'toolenv' / 'tools')
lookup = build_api_lookup(corpus)
evals = load_eval_examples(TOOLBENCH_DIR / 'toolllama_G123_dfs_eval.json')
with open(PROJECT_DIR / 'api_names.json') as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}

print(f'Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples')

## 1. Additional Embedding Models

Compare `text-embedding-3-small` (1536-dim) with `text-embedding-3-large` (3072-dim) to test whether the semantic gap is model-specific or fundamental.

In [ ]:
LARGE_MODEL = 'text-embedding-3-large'

# Embed all APIs with the large model
from tqdm import tqdm

api_strings = [format_api_string(a) for a in corpus]

BATCH = 500
large_embeddings = []
for i in tqdm(range(0, len(api_strings), BATCH), desc='Embedding APIs (large)'):
    embs = get_embeddings(api_strings[i:i+BATCH], model=LARGE_MODEL)
    large_embeddings.append(embs)
large_matrix = np.vstack(large_embeddings)

# Embed queries
queries = [e['user_query'] for e in evals]
large_query_embs = get_embeddings(queries, model=LARGE_MODEL)

print(f'API embeddings: {large_matrix.shape}, Query embeddings: {large_query_embs.shape}')

In [ ]:
# Full-corpus evaluation with large model
large_index = build_faiss_index(large_matrix)
all_retrieved, all_gt = [], []
for i, ex in enumerate(evals):
    top_k = retrieve_top_k(large_query_embs[i], large_index, k=10)
    all_retrieved.append(top_k)
    all_gt.append([name_to_idx[n] for n in ex['ground_truth_apis'] if n in name_to_idx])

large_results = evaluate_batch(all_retrieved, all_gt, ks=[1, 5, 10])

# Load small model results for comparison
with open(PROJECT_DIR / 'results_baseline.json') as f:
    small_results = json.load(f)['results']

print(f"{'Model':<30} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 62)
for label, r in [('text-embedding-3-small', small_results), ('text-embedding-3-large', large_results)]:
    print(f"{label:<30} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

In [ ]:
# Hard-negative ablation with large model
with open(TOOLBENCH_DIR / 'toolllama_G123_dfs_eval.json') as f:
    raw_evals = json.load(f)

def eval_restricted_pool(query_embs, corpus_matrix, neg_type, n_neg=99, ks=[1, 5, 10]):
    recall_scores = {k: [] for k in ks}
    mrr_scores = []
    for i, ex in enumerate(evals):
        raw_ex = raw_evals[ex['raw_idx']]
        gt_names = ex['ground_truth_apis']
        gt_indices = [name_to_idx[n] for n in gt_names if n in name_to_idx]
        if not gt_indices:
            continue
        if neg_type == 'random':
            negs = build_random_negatives(corpus, gt_names, n=n_neg)
        elif neg_type == 'sibling':
            negs = build_category_sibling_negatives(corpus, gt_names, lookup, n=n_neg)
        else:
            negs = build_dfsdt_negatives(raw_ex, corpus, gt_names, lookup, n=n_neg)
        cand_names = gt_names + [a['action_name'] for a in negs]
        cand_indices = [name_to_idx[n] for n in cand_names if n in name_to_idx]
        if len(cand_indices) < 2:
            continue
        local_index = build_faiss_index(corpus_matrix[cand_indices].astype(np.float32))
        g2l = {g: j for j, g in enumerate(cand_indices)}
        local_gt = [g2l[g] for g in gt_indices if g in g2l]
        if not local_gt:
            continue
        top_k = retrieve_top_k(query_embs[i], local_index, k=min(10, len(cand_indices)))
        for k in ks:
            recall_scores[k].append(recall_at_k(top_k, local_gt, k))
        mrr_scores.append(mean_reciprocal_rank(top_k, local_gt))
    return {**{f'recall@{k}': float(np.mean(recall_scores[k])) for k in ks}, 'mrr': float(np.mean(mrr_scores))}

large_hn = {}
for neg_type in ['random', 'sibling', 'dfsdt']:
    print(f'Running: {neg_type}...')
    large_hn[neg_type] = eval_restricted_pool(large_query_embs, large_matrix, neg_type)

with open(PROJECT_DIR / 'results_hard_negatives.json') as f:
    small_hn = json.load(f)

print(f"\n{'Model + Condition':<45} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 77)
for model, hn in [('small', small_hn), ('large', large_hn)]:
    for cond in ['random', 'sibling', 'dfsdt']:
        r = hn[cond]
        print(f"text-embedding-3-{model} / {cond:<15} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

## 2. API Format Ablation

Test whether including category, tool name, and parameters in the API string helps retrieval, or if the model only uses the name and description.

In [ ]:
# Embed queries once (shared across all format modes)
queries = [e['user_query'] for e in evals]
query_embs = get_embeddings(queries)

format_results = {}
for mode in ['full', 'name_desc', 'desc_only', 'json']:
    print(f'Format: {mode}...')
    strings = [format_api_string(a, mode=mode) for a in corpus]
    embs = []
    for i in tqdm(range(0, len(strings), BATCH), desc=mode):
        embs.append(get_embeddings(strings[i:i+BATCH]))
    matrix = np.vstack(embs)
    index = build_faiss_index(matrix)

    all_retrieved, all_gt = [], []
    for j, ex in enumerate(evals):
        top_k = retrieve_top_k(query_embs[j], index, k=10)
        all_retrieved.append(top_k)
        all_gt.append([name_to_idx[n] for n in ex['ground_truth_apis'] if n in name_to_idx])
    format_results[mode] = evaluate_batch(all_retrieved, all_gt, ks=[1, 5, 10])

print(f"\n{'Format':<15} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 47)
for mode, r in format_results.items():
    print(f"{mode:<15} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

## 3. Error Analysis

Analyze where the baseline fails: by category, query complexity, and qualitative examples.

In [ ]:
# Re-run baseline retrieval (or reuse if already in memory)
corpus_matrix = np.load(PROJECT_DIR / 'corpus_embeddings.npy')
query_embs_small = get_embeddings([e['user_query'] for e in evals])
index = build_faiss_index(corpus_matrix)

per_example = []
for i, ex in enumerate(evals):
    top_k = retrieve_top_k(query_embs_small[i], index, k=10)
    gt_indices = [name_to_idx[n] for n in ex['ground_truth_apis'] if n in name_to_idx]
    r1 = recall_at_k(top_k, gt_indices, 1)
    r5 = recall_at_k(top_k, gt_indices, 5)
    mrr = mean_reciprocal_rank(top_k, gt_indices)
    gt_cats = {lookup[n]['category'] for n in ex['ground_truth_apis'] if n in lookup}
    per_example.append({
        'idx': i, 'r1': r1, 'r5': r5, 'mrr': mrr,
        'n_gt': len(gt_indices), 'query_len': len(ex['user_query']),
        'categories': gt_cats, 'top_k': top_k, 'gt_indices': gt_indices,
        'query': ex['user_query'], 'gt_names': ex['ground_truth_apis'],
    })

In [ ]:
# 3a. Per-category R@5
from collections import defaultdict

cat_scores = defaultdict(list)
for ex in per_example:
    for cat in ex['categories']:
        cat_scores[cat].append(ex['r5'])

cat_r5 = {cat: np.mean(scores) for cat, scores in cat_scores.items() if len(scores) >= 5}
sorted_cats = sorted(cat_r5.items(), key=lambda x: x[1])

print(f"{'Category':<40} {'R@5':>8} {'Count':>8}")
print('-' * 56)
for cat, score in sorted_cats[:10]:
    print(f"{cat[:40]:<40} {score:>8.3f} {len(cat_scores[cat]):>8}")
print('...')
for cat, score in sorted_cats[-5:]:
    print(f"{cat[:40]:<40} {score:>8.3f} {len(cat_scores[cat]):>8}")

fig, ax = plt.subplots(figsize=(10, 5))
cats_plot = sorted_cats[:15]
ax.barh([c[:30] for c, _ in cats_plot], [s for _, s in cats_plot], color='steelblue')
ax.set_xlabel('Recall@5')
ax.set_title('Hardest Categories (baseline)')
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'error_by_category.png', dpi=150)
plt.show()

In [ ]:
# 3b. By number of ground-truth APIs and query length
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Group by n_gt
gt_groups = defaultdict(list)
for ex in per_example:
    gt_groups[ex['n_gt']].append(ex['r5'])
gt_sorted = sorted(gt_groups.items())
axes[0].bar([str(k) for k, _ in gt_sorted], [np.mean(v) for _, v in gt_sorted], color='darkorange')
axes[0].set_xlabel('Number of ground-truth APIs')
axes[0].set_ylabel('Recall@5')
axes[0].set_title('R@5 by query complexity')

# Group by query length quartile
lengths = np.array([ex['query_len'] for ex in per_example])
quartiles = np.percentile(lengths, [25, 50, 75])
def qbin(l):
    if l <= quartiles[0]: return 'Q1 (short)'
    if l <= quartiles[1]: return 'Q2'
    if l <= quartiles[2]: return 'Q3'
    return 'Q4 (long)'

len_groups = defaultdict(list)
for ex in per_example:
    len_groups[qbin(ex['query_len'])].append(ex['r5'])
for label in ['Q1 (short)', 'Q2', 'Q3', 'Q4 (long)']:
    axes[1].bar(label, np.mean(len_groups[label]), color='firebrick')
axes[1].set_ylabel('Recall@5')
axes[1].set_title('R@5 by query length quartile')

plt.tight_layout()
plt.savefig(PROJECT_DIR / 'error_by_complexity.png', dpi=150)
plt.show()

In [ ]:
# 3c. Worst failure examples
worst = sorted(per_example, key=lambda x: x['mrr'])[:5]

for rank, ex in enumerate(worst, 1):
    print(f"\n=== Failure #{rank} (MRR={ex['mrr']:.3f}, R@5={ex['r5']:.3f}) ===")
    print(f"Query: {ex['query'][:200]}")
    print(f"Expected: {ex['gt_names'][:3]}")
    retrieved_names = [api_names[j] for j in ex['top_k'][:5]]
    print(f"Retrieved: {retrieved_names}")

In [ ]:
# Save all Phase 1 results
phase1_results = {
    'model_comparison': {
        'text-embedding-3-small': small_results,
        'text-embedding-3-large': large_results,
    },
    'format_ablation': format_results,
    'per_category_r5': {cat: float(score) for cat, score in sorted_cats},
}
with open(PROJECT_DIR / 'results_phase1.json', 'w') as f:
    json.dump(phase1_results, f, indent=2)
print('Saved results_phase1.json')